# getting medical simplifcation data from github 

In [1]:
import requests
import os
import datetime
import random
import zipfile
import shutil
import math
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW # Changed from transformers import AdamW
import datasets as hf_datasets # Alias to avoid conflict with local variables
import transformers as hf_transformers # Alias
from datasets import Dataset, DatasetDict, load_dataset
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, set_seed, get_scheduler
from accelerate import Accelerator, notebook_launcher
from tqdm.notebook import tqdm # Or from tqdm import tqdm
import importlib.metadata # For version checking


import error: No module named 'triton'


In [2]:

user = "SebaJoe"
repo = "MultiCochrane"
branch = "main" 
path_to_dir = "data/MultiCochrane"

filenames = [
    "multiCochrane_all.zip"
]

local_dir = "./data"

try:
    os.makedirs(local_dir, exist_ok=True)
    print(f"Directory '{local_dir}' ensured.")
except OSError as e:
    print(f"Error creating directory {local_dir}: {e}")
    filenames = [] # Prevent download attempts if dir fails

download_count = 0
error_count = 0
for filename in filenames:
    # Construct the URL to the raw file content
    raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{path_to_dir}/{filename}"

    # Construct the full local path
    local_filepath = os.path.join(local_dir, filename)

    print(f"  Downloading '{filename}' from {raw_url}...")

    try:
        response = requests.get(raw_url, stream=True) 
        response.raise_for_status() 

        # Write the content to the local file
        with open(local_filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"  Successfully saved to '{local_filepath}'")
        download_count += 1

    except requests.exceptions.RequestException as e:
        print(f"  Error downloading {filename}: {e}")
        error_count += 1
    except IOError as e:
         print(f"  Error writing file {local_filepath}: {e}")
         error_count += 1
    except Exception as e:
        print(f"  An unexpected error occurred for {filename}: {e}")
        error_count += 1


print("\n--- Download Summary ---")
print(f"Successfully downloaded: {download_count} file(s)")
print(f"Errors encountered: {error_count} file(s)")
if error_count == 0 and download_count == len(filenames):
    print("All expected files downloaded successfully.")
elif download_count > 0:
     print("Some files downloaded. Check logs for errors.")
else:
     print("No files were downloaded.")

Directory './data' ensured.
  Successfully saved to './data\multiCochrane_all.zip'

--- Download Summary ---
Successfully downloaded: 1 file(s)
Errors encountered: 0 file(s)
All expected files downloaded successfully.


# Loading the data

In [3]:
#for the new data (muilticochrane_all.zip)
def unzip_file(zip_file_path, extract_to_dir):
    """
    Unzips a specified zip file to a target directory.

    Args:
        zip_file_path (str): The path to the .zip file to be extracted.
        extract_to_dir (str): The directory where the contents should be extracted.
                               If it doesn't exist, it will be created.
    """
    try:
        os.makedirs(extract_to_dir, exist_ok=True)
        print(f"Ensured extraction directory '{extract_to_dir}' exists.")
    except OSError as e:
        print(f"Error creating directory {extract_to_dir}: {e}")
        return # Stop if directory creation fails

    if not os.path.isfile(zip_file_path):
        print(f"Error: Zip file not found at '{zip_file_path}'")
        return

    print(f"Attempting to extract '{zip_file_path}' to '{extract_to_dir}'...")
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to_dir)
        print(f"Successfully extracted '{zip_file_path}' to '{extract_to_dir}'")

    except zipfile.BadZipFile:
        print(f"Error: Failed to unzip. '{zip_file_path}' might be corrupted or not a valid zip file.")
    except FileNotFoundError:
        print(f"Error: Zip file not found at '{zip_file_path}' during opening.")
    except Exception as e:
        print(f"An unexpected error occurred during extraction: {e}")


path_to_my_zip = local_dir + "/" + filenames[0] 
destination_folder = local_dir + "/multiCochrane_all"
unzip_file(path_to_my_zip, destination_folder)

Ensured extraction directory './data/multiCochrane_all' exists.
Attempting to extract './data/multiCochrane_all.zip' to './data/multiCochrane_all'...
Successfully extracted './data/multiCochrane_all.zip' to './data/multiCochrane_all'


In [4]:
base_path = "./data/multiCochrane_all/filtered (r=0.5)/en"
data_files = {
    "train": os.path.join(base_path, "train0.5_en.csv"),
    "test": os.path.join(base_path, "test0.5_en.csv"),
    "validation": os.path.join(base_path, "val0.5_en.csv") 
}

try:
    multi_cochrane_dataset = load_dataset("csv", data_files=data_files)
    print(multi_cochrane_dataset)
except Exception as e:
    print(f"\nAn error occurred during dataset loading: {e}")


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 30032
    })
    test: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 395
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 83
    })
})


In [5]:
multi_cochrane_dataset["train"][6]

{'Unnamed: 0': 15,
 'prefix': None,
 'input_text': 'We could not say whether more people who had a remote check-up needed oral corticosteroids for an asthma exacerbation than those who were seen face-to-face because the confidence intervals (CIs) were very wide (OR 1.74, 95% CI 0.41 to 7.44; 278 participants; one study; low quality evidence).',
 'target_text': 'We cannot say whether or not people who had a check-up over the phone or internet were more or less likely to need oral corticosteroids for an asthma attack than those seen face-to-face, and we were uncertain of the result for several reasons.',
 'doi': '10.1002/14651858.CD011715.pub2'}

# Encode dataset


In [6]:
#based on https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/T5/Fine_tuning_Dutch_T5_base_on_CNN_Daily_Mail_for_summarization_(on_TPU_using_HuggingFace_Accelerate).ipynb#scrollTo=tiLdcTmkg-_o
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-base")

prefix = "Simplify: "
max_input_length = 512
max_target_length = 512

def preprocess_examples(examples):
  # encode the documents
  input_txt = examples['input_text']
  target_txt = examples['target_text']
  
  inputs = [prefix + inp for inp in input_txt]
  model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

  # encode the simplifications
  labels = tokenizer(target_txt, max_length=max_target_length, padding="max_length", truncation=True).input_ids

  # important: we need to replace the index of the padding tokens by -100
  # such that they are not taken into account by the CrossEntropyLoss
  labels_with_ignore_index = []
  for labels_example in labels:
    labels_example = [label if label != 0 else -100 for label in labels_example]
    labels_with_ignore_index.append(labels_example)
  
  model_inputs["labels"] = labels_with_ignore_index

  return model_inputs

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [7]:
train_ds = multi_cochrane_dataset['train']
val_ds = multi_cochrane_dataset['validation']
test_ds = multi_cochrane_dataset['test']
encoded_train_ds = train_ds.map(preprocess_examples, batched=True, remove_columns=train_ds.column_names)
encoded_val_ds = val_ds.map(preprocess_examples, batched=True, remove_columns=val_ds.column_names)
encoded_test_ds = test_ds.map(preprocess_examples, batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/30032 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

In [8]:
#set format to PyTorch
encoded_train_ds.set_format(type="torch")
encoded_val_ds.set_format(type="torch")
encoded_test_ds.set_format(type="torch")

In [9]:
from torch.utils.data import DataLoader

def create_dataloaders(train_batch_size=8, eval_batch_size=32):
    train_dataloader = DataLoader(encoded_train_ds, shuffle=True, batch_size=train_batch_size)
    val_dataloader = DataLoader(encoded_val_ds, shuffle=False, batch_size=eval_batch_size)
    return train_dataloader, val_dataloader

# FineTuning model

In [10]:
hyperparameters = {
    "model_checkpoint": "google-t5/t5-small",
    "learning_rate": 5e-5, 
    "num_epochs": 3, 
    "train_batch_size": 8, 
    "gradient_accumulation_steps": 4,
    "eval_batch_size": 16, 
    "seed": 42,
    "patience": 3,
    "output_dir": "./content_simplifier_t5_small_new_data/",
    "mixed_precision": "fp16",
}

In [11]:
import torch
from transformers import T5ForConditionalGeneration, AdamW, set_seed
from accelerate import Accelerator
from tqdm.notebook import tqdm
import datasets
import transformers


def training_function():
    # Initialize accelerator
    accelerator = Accelerator()

    # To have only one message (and not 8) per logs of Transformers or Datasets, we set the logging verbosity
    # to INFO for the main process only.
    if accelerator.is_main_process:
        datasets.utils.logging.set_verbosity_warning()
        transformers.utils.logging.set_verbosity_info()
    else:
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()

    # The seed need to be set before we instantiate the model, as it will determine the random head.
    set_seed(hyperparameters["seed"])

    # Instantiate the model, let Accelerate handle the device placement.
    model = T5ForConditionalGeneration.from_pretrained(hyperparameters["model_checkpoint"])

    # Instantiate optimizer
    optimizer = AdamW(model.parameters(), lr=hyperparameters["learning_rate"])

    # Prepare everything
    train_dataloader, val_dataloader = create_dataloaders(
        train_batch_size=hyperparameters["train_batch_size"], eval_batch_size=hyperparameters["eval_batch_size"]
    )
    # There is no specific order to remember, we just need to unpack the objects in the same order we gave them to the
    # prepare method.
    model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(model, optimizer, 
                                                                             train_dataloader, val_dataloader)
    
    # Now we train the model
    epochs_no_improve = 0
    min_val_loss = 1000000
    for epoch in range(hyperparameters["num_epochs"]):
        # We only enable the progress bar on the main process to avoid having 8 progress bars.
        progress_bar = tqdm(range(len(train_dataloader)), disable=not accelerator.is_main_process)
        progress_bar.set_description(f"Epoch: {epoch}")
        model.train()
        for batch in train_dataloader:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            
            optimizer.step()
            optimizer.zero_grad()
            progress_bar.set_postfix({'loss': loss.item()})
            progress_bar.update(1)

        # Evaluate at the end of the epoch (distributed evaluation as we have 8 TPU cores)
        model.eval()
        validation_losses = []
        for batch in val_dataloader:
            with torch.no_grad():
                outputs = model(**batch)
            loss = outputs.loss

            # We gather the loss from the 8 TPU cores to have them all.
            validation_losses.append(accelerator.gather(loss[None]))

        # Compute average validation loss
        val_loss = torch.stack(validation_losses).sum().item() / len(validation_losses)
        # Use accelerator.print to print only on the main process.
        accelerator.print(f"epoch {epoch}: validation loss:", val_loss)
        if val_loss < min_val_loss:
          epochs_no_improve = 0
          min_val_loss = val_loss
          continue
        else:
          epochs_no_improve += 1
          # Check early stopping condition
          if epochs_no_improve == hyperparameters["patience"]:
            accelerator.print("Early stopping!")
            break

    # save trained model
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    # Use accelerator.save to save
    unwrapped_model.save_pretrained(hyperparameters["output_dir"], save_function=accelerator.save)

In [12]:
training_function()

loading configuration file config.json from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-small\snapshots\df1b051c49625cf57a3d0d8d3863ed4d13564fe4\config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_n

  0%|          | 0/3754 [00:00<?, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


epoch 0: validation loss: 2.2850608825683594


  0%|          | 0/3754 [00:00<?, ?it/s]

epoch 1: validation loss: 2.2042312622070312


  0%|          | 0/3754 [00:00<?, ?it/s]

Configuration saved in ./content_simplifier_t5_small_new_data/config.json
Configuration saved in ./content_simplifier_t5_small_new_data/generation_config.json
Model weights saved in ./content_simplifier_t5_small_new_data/model.safetensors


epoch 2: validation loss: 2.173359235127767


In [ ]:
text = """Simplify: The patient presented with refractory ventricular tachycardia""" 

trained_model = T5ForConditionalGeneration.from_pretrained(r"C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\code\content_simplifier_t5_base_simple")

input_ids = tokenizer(text, return_tensors="pt").input_ids
 
generated_ids = trained_model.generate(input_ids, do_sample=True, 
    max_length=50, 
    top_k=4, 
    temperature=0.7
)

summary = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print(summary)

loading configuration file C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\code\content_simplifier_t5_base_simple\config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      

Original Text: The patient presented with refractory ventricular tachycardia.
Simplified: sss


In [21]:
print(summary)

The patient presented with refractory ventricular tachycardia (VT) and ventricular tachycardia (VT) in a refractory patient.


In [7]:
import os
import torch
import random
import datasets
import transformers
import math
import evaluate # Added for BLEU

from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed,
    get_scheduler # Although imported, not explicitly used in the provided training loop logic
)
from torch.optim import AdamW
from torch.utils.data import DataLoader
from accelerate import Accelerator
from tqdm.notebook import tqdm
import numpy as np # Needed for label processing before decoding

# --- Dataset Loading ---
base_path = "./data/multiCochrane_all/filtered (r=0.5)/en"
data_files = {
    "train": os.path.join(base_path, "train0.5_en.csv"),
    "validation": os.path.join(base_path, "val0.5_en.csv"),
    "test": os.path.join(base_path, "test0.5_en.csv")
}

try:
    multi_cochrane_dataset = load_dataset("csv", data_files=data_files)
    print("Dataset loaded successfully:")
    print(multi_cochrane_dataset)
except Exception as e:
    print(f"\nAn error occurred during dataset loading: {e}")
    raise

print("\nSample from training data:")
print("input_text:", multi_cochrane_dataset["train"][0]["input_text"])
print("target_text:", multi_cochrane_dataset["train"][0]["target_text"])

# --- Tokenizer & Preprocessing ---
model_checkpoint = "google-t5/t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)

prefix = "Simplify: "
max_input_length = 128
max_target_length = 128

def preprocess_examples(examples):
    input_txt = examples["input_text"]
    target_txt = examples["target_text"]

    inputs = [prefix + inp for inp in input_txt]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        padding="max_length",
        truncation=True
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            target_txt,
            max_length=max_target_length,
            padding="max_length",
            truncation=True
        )["input_ids"]

    labels_with_ignore_index = []
    for label_example in labels:
        label_example = [l if l != tokenizer.pad_token_id else -100 for l in label_example]
        labels_with_ignore_index.append(label_example)

    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

# --- Preprocess and Encode Dataset ---
train_ds = multi_cochrane_dataset["train"]
val_ds = multi_cochrane_dataset["validation"]
test_ds = multi_cochrane_dataset["test"]

encoded_train_ds = train_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=train_ds.column_names
)
encoded_val_ds = val_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=val_ds.column_names
)
encoded_test_ds = test_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=test_ds.column_names
)

encoded_train_ds.set_format(type="torch")
encoded_val_ds.set_format(type="torch")
encoded_test_ds.set_format(type="torch")

# --- Dataloaders ---
def create_dataloaders(train_batch_size=32, eval_batch_size=32):
    train_dataloader = DataLoader(encoded_train_ds, shuffle=True, batch_size=train_batch_size)
    val_dataloader   = DataLoader(encoded_val_ds, shuffle=False, batch_size=eval_batch_size)
    return train_dataloader, val_dataloader

# --- Hyperparameters ---
hyperparameters = {
    "model_checkpoint": model_checkpoint,
    "learning_rate": 5e-5,
    "num_epochs": 5,
    "train_batch_size": 64,
    "gradient_accumulation_steps": 4, # Not used in the provided training loop
    "eval_batch_size": 64,
    "seed": 42,
    "patience": 3,
    "output_dir": "./content_simplifier_t5_small_sari/", # Updated output dir name
    "mixed_precision": "fp16",
    "eval_generation_max_length": max_target_length # Added for generation during eval
}

# --- Training Function ---
def training_function():
    accelerator = Accelerator(mixed_precision=hyperparameters["mixed_precision"])

    if accelerator.is_main_process:
        datasets.utils.logging.set_verbosity_warning()
        transformers.utils.logging.set_verbosity_info()
    else:
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()

    set_seed(hyperparameters["seed"])

    model = T5ForConditionalGeneration.from_pretrained(hyperparameters["model_checkpoint"])

    optimizer = AdamW(model.parameters(), lr=hyperparameters["learning_rate"])

    train_dataloader, val_dataloader = create_dataloaders(
        train_batch_size=hyperparameters["train_batch_size"],
        eval_batch_size=hyperparameters["eval_batch_size"]
    )

    model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, val_dataloader
    )

    # Load BLEU metric
    bleu_metric = evaluate.load("bleu")

    epochs_no_improve = 0
    # For BLEU, higher is better, so we track max score
    max_val_bleu = 0.0

    num_training_steps = len(train_dataloader) * hyperparameters["num_epochs"] # Calculate total steps if using scheduler

    for epoch in range(hyperparameters["num_epochs"]):
        progress_bar_train = tqdm(range(len(train_dataloader)), disable=not accelerator.is_main_process)
        progress_bar_train.set_description(f"Epoch {epoch} Training")

        # --- Training ---
        model.train()
        total_loss = 0
        for batch in train_dataloader:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)

            optimizer.step()
            # lr_scheduler.step() # Uncomment if using a learning rate scheduler
            optimizer.zero_grad()
            total_loss += loss.item()
            progress_bar_train.set_postfix({'loss': loss.item()})
            progress_bar_train.update(1)
        avg_train_loss = total_loss / len(train_dataloader)
        accelerator.print(f"Epoch {epoch}: Average Training Loss = {avg_train_loss:.4f}")


        # --- Evaluation (using BLEU) ---
        model.eval()
        progress_bar_eval = tqdm(range(len(val_dataloader)), disable=not accelerator.is_main_process)
        progress_bar_eval.set_description(f"Epoch {epoch} Evaluation")

        all_preds = []
        all_labels = []

        for batch in val_dataloader:
            with torch.no_grad():
                # Generate predictions
                generated_tokens = accelerator.unwrap_model(model).generate(
                    batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_length=hyperparameters["eval_generation_max_length"],
                    # Add other generation parameters like num_beams if desired
                )
            # Gather predictions and labels across devices
            generated_tokens = accelerator.gather_for_metrics(generated_tokens)
            labels = batch["labels"]
            # Replace -100 with pad_token_id for decoding
            labels = labels.cpu().numpy()
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            labels = accelerator.gather_for_metrics(torch.tensor(labels).to(generated_tokens.device)) # Ensure labels are gathered too

            # Decode predictions and labels
            decoded_preds = tokenizer.batch_decode(generated_tokens.cpu().numpy(), skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels.cpu().numpy(), skip_special_tokens=True)

            # Add to BLEU metric - Need list of lists for references
            bleu_metric.add_batch(predictions=decoded_preds, references=[[label] for label in decoded_labels])
            progress_bar_eval.update(1)

        # Compute BLEU score
        eval_metric = bleu_metric.compute()
        val_bleu = eval_metric["bleu"] # Extract the BLEU score

        accelerator.print(f"Epoch {epoch}: Validation BLEU = {val_bleu:.4f}")

        # --- Early stopping check based on BLEU ---
        if val_bleu > max_val_bleu:
            epochs_no_improve = 0
            max_val_bleu = val_bleu
            accelerator.print(f"New best BLEU score: {max_val_bleu:.4f}. Saving model.")
            # Save model only when BLEU improves
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(
                hyperparameters["output_dir"],
                save_function=accelerator.save,
                is_main_process=accelerator.is_main_process # Ensure only main process saves config etc.
            )
            # Also save tokenizer with the model
            if accelerator.is_main_process:
                 tokenizer.save_pretrained(hyperparameters["output_dir"])

        else:
            epochs_no_improve += 1
            if epochs_no_improve >= hyperparameters["patience"]:
                accelerator.print(f"No improvement in BLEU for {hyperparameters['patience']} epochs. Early stopping triggered.")
                break


training_function()


# --- Inference ---
trained_model_path = hyperparameters["output_dir"] # Use the updated output dir
trained_model = T5ForConditionalGeneration.from_pretrained(trained_model_path)

# Move model to appropriate device for inference (optional, but good practice)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model.to(device)

sample_text = "The patient presented with refractory ventricular tachycardia."
prompt = "Simplify: " + sample_text

input_ids =  tokenizer(prompt, return_tensors="pt").input_ids.to(device) # Move input to device

generated_ids = trained_model.generate(
    input_ids=input_ids,
    max_length=50,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print("\n--- Inference Example ---")
print("Original Text:", sample_text)
print("Simplified:", summary)

loading file spiece.model from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-small\snapshots\df1b051c49625cf57a3d0d8d3863ed4d13564fe4\spiece.model
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-small\snapshots\df1b051c49625cf57a3d0d8d3863ed4d13564fe4\tokenizer_config.json
loading file tokenizer.json from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-small\snapshots\df1b051c49625cf57a3d0d8d3863ed4d13564fe4\tokenizer.json
loading file chat_template.jinja from cache at None


Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 30032
    })
    test: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 395
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 83
    })
})

Sample from training data:
input_text: Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates, but at present, the evidence is very limited with data rated as very low quality.
target_text: However, at the moment evidence is very limited with data only of very low quality available.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/30032 [00:00<?, ? examples/s]

c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


...

  0%|          | 0/2 [00:00<?, ?it/s]

loading configuration file ./content_simplifier_t5_small_sari/config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
    

Epoch 4: Validation BLEU = 0.1986


All model checkpoint weights were used when initializing T5ForConditionalGeneration.

All the weights of T5ForConditionalGeneration were initialized from the model checkpoint at ./content_simplifier_t5_small_sari/.
If your task is similar to the task the model of the checkpoint was trained on, you can already use T5ForConditionalGeneration for predictions without further training.
loading configuration file ./content_simplifier_t5_small_sari/generation_config.json
Generate config GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0
}




--- Inference Example ---
Original Text: The patient presented with refractory ventricular tachycardia.
Simplified: The patient presented with refractory ventricular tachycardia.


In [6]:
train_ds[0]

{'Unnamed: 0': 0,
 'prefix': None,
 'input_text': 'Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates, but at present, the evidence is very limited with data rated as very low quality.',
 'target_text': 'However, at the moment evidence is very limited with data only of very low quality available.',
 'doi': '10.1002/14651858.CD009006.pub2'}

In [ ]:
import os
import torch
import datasets
import transformers
from datasets import load_dataset
import evaluate
from transformers import T5Tokenizer, T5ForConditionalGeneration, set_seed, AdamW
from torch.utils.data import DataLoader
from accelerate import Accelerator
from tqdm.notebook import tqdm

# Data loading
base_path = "./data/multiCochrane_all/filtered (r=0.5)/en"
data_files = {
    "train": os.path.join(base_path, "train0.5_en.csv"),
    "validation": os.path.join(base_path, "val0.5_en.csv"),
    "test": os.path.join(base_path, "test0.5_en.csv")
}
multi_cochrane_dataset = load_dataset("csv", data_files=data_files)

# Model and tokenizer setup
model_checkpoint = "google-t5/t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)
prefix = "Simplify: "
max_input_length = 256
max_target_length = 256

# Preprocessing function
def preprocess_examples(examples):
    input_txt = examples["input_text"]
    target_txt = examples["target_text"]
    inputs = [prefix + inp for inp in input_txt]
    model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target_txt, max_length=max_target_length, padding="max_length", truncation=True)["input_ids"]
    # Replace pad_token_id with -100 for loss computation
    labels_with_ignore_index = [[l if l != tokenizer.pad_token_id else -100 for l in label_example] for label_example in labels]
    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

# Dataset preparation
train_ds = multi_cochrane_dataset["train"]
val_ds = multi_cochrane_dataset["validation"]
test_ds = multi_cochrane_dataset["test"]

encoded_train_ds = train_ds.map(preprocess_examples, batched=True, remove_columns=train_ds.column_names)
encoded_val_ds = val_ds.map(preprocess_examples, batched=True, remove_columns=val_ds.column_names)
encoded_test_ds = test_ds.map(preprocess_examples, batched=True, remove_columns=test_ds.column_names)

encoded_train_ds.set_format(type="torch")
encoded_val_ds.set_format(type="torch")
encoded_test_ds.set_format(type="torch")

# DataLoader creation
def create_dataloaders(train_batch_size=32, eval_batch_size=32):
    train_dataloader = DataLoader(encoded_train_ds, shuffle=True, batch_size=train_batch_size)
    val_dataloader = DataLoader(encoded_val_ds, shuffle=False, batch_size=eval_batch_size)
    return train_dataloader, val_dataloader

# Hyperparameters
hyperparameters = {
    "model_checkpoint": model_checkpoint,
    "learning_rate": 5e-5,
    "num_epochs": 2,
    "train_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "eval_batch_size": 16,
    "seed": 42,
    "patience": 3,
    "output_dir": "./content_simplifier_t5_small_simple/",
    "mixed_precision": "fp16"
}

# Training function
def training_function():
    accelerator = Accelerator(mixed_precision=hyperparameters["mixed_precision"])
    if accelerator.is_main_process:
        datasets.utils.logging.set_verbosity_warning()
        transformers.utils.logging.set_verbosity_info()
    else:
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()
    set_seed(hyperparameters["seed"])
    
    model = T5ForConditionalGeneration.from_pretrained(hyperparameters["model_checkpoint"])
    optimizer = AdamW(model.parameters(), lr=hyperparameters["learning_rate"])
    train_dataloader, val_dataloader = create_dataloaders(
        train_batch_size=hyperparameters["train_batch_size"],
        eval_batch_size=hyperparameters["eval_batch_size"]
    )
    model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, val_dataloader
    )
    
    epochs_no_improve = 0
    best_sari = 0.0
    sari_metric = evaluate.load("sari")
    
    for epoch in range(hyperparameters["num_epochs"]):
        # Training loop
        progress_bar = tqdm(range(len(train_dataloader)), disable=not accelerator.is_main_process)
        progress_bar.set_description(f"Epoch {epoch}")
        model.train()
        for batch in train_dataloader:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()
            progress_bar.set_postfix({'loss': loss.item()})
            progress_bar.update(1)
        
        # Evaluation loop
        model.eval()
        all_sources = []
        all_predictions = []
        all_references = []
        for batch in val_dataloader:
            with torch.no_grad():
                generated_ids = accelerator.unwrap_model(model).generate(
                    input_ids=batch["input_ids"],
                    max_length=max_target_length,
                    num_beams=4,
                    early_stopping=True
                )
                generated_texts = accelerator.gather(generated_ids)
                labels = accelerator.gather(batch["labels"])
                input_ids = accelerator.gather(batch["input_ids"])

                if accelerator.is_main_process:
                    generated_texts = generated_texts.cpu().numpy()
                    labels = labels.cpu().numpy()
                    input_ids = input_ids.cpu().numpy()

                    # Fix: Replace -100 with pad_token_id before decoding
                    labels[labels == -100] = tokenizer.pad_token_id

                    # Decode predictions, labels, and inputs
                    decoded_preds = tokenizer.batch_decode(generated_texts, skip_special_tokens=True)
                    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
                    decoded_inputs = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
                    # Remove prefix to get original source texts
                    sources = [text.replace("Simplify: ", "", 1) for text in decoded_inputs]

                    # Collect for SARI computation
                    all_predictions.extend(decoded_preds)
                    all_references.extend([[label] for label in decoded_labels])  # SARI expects list of lists
                    all_sources.extend(sources)

        if accelerator.is_main_process:
            # Compute SARI score
            results = sari_metric.compute(sources=all_sources, predictions=all_predictions, references=all_references)
            sari_score = results["sari"]
            accelerator.print(f"Epoch {epoch}: SARI = {sari_score:.4f}")
            
            # Early stopping based on SARI
            if sari_score > best_sari:
                best_sari = sari_score
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= hyperparameters["patience"]:
                    accelerator.print("Early stopping triggered.")
                    break
    
    # Save the model
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(hyperparameters["output_dir"], save_function=accelerator.save)

# Run training
training_function()


import error: No module named 'triton'


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
loading configuration file config.json from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-small\snapshots\df1b051c49625cf57a3d0d8d3863ed4d13564fe4\config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is

  0%|          | 0/3754 [00:00<?, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 0: SARI = 43.5024


  0%|          | 0/3754 [00:00<?, ?it/s]

Configuration saved in ./content_simplifier_t5_small_simple/config.json
Configuration saved in ./content_simplifier_t5_small_simple/generation_config.json


Epoch 1: SARI = 43.8924


Model weights saved in ./content_simplifier_t5_small_simple/model.safetensors
loading configuration file ./content_simplifier_t5_small_simple/config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_be

OSError: Can't load tokenizer for './content_simplifier_t5_small_simple/'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './content_simplifier_t5_small_simple/' is the correct path to a directory containing all relevant files for a T5Tokenizer tokenizer.

In [ ]:

# Load trained model and generate a sample simplification
trained_model_path = hyperparameters["output_dir"]
trained_model = T5ForConditionalGeneration.from_pretrained(trained_model_path)

sample_text = "The patient presented with severe issues relating to a refractory ventricular tachycardia."
prompt = "Simplify: " + sample_text
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
generated_ids = trained_model.generate(input_ids=input_ids, max_length=50, num_beams=4, early_stopping=True)
summary = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print("Original Text:", sample_text)
print("Simplified:", summary)

loading configuration file ./content_simplifier_t5_small_simple/config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
  

Original Text: The patient presented with severe issues relating to a refractory ventricular tachycardia.
Simplified: The patient presented with severe issues relating to a refractory ventricular tachycardia.


In [4]:
sample_text = "The patient presented with severe humongous issues relating to a refractory ventricular tachycardia."
prompt = "Simplify: " + sample_text
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
generated_ids = trained_model.generate(input_ids=input_ids, max_length=50, num_beams=4, early_stopping=True)
summary = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print("Original Text:", sample_text)
print("Simplified:", summary)

Original Text: The patient presented with severe humongous issues relating to a refractory ventricular tachycardia.
Simplified: The patient presented with severe humongous issues relating to a refractory ventricular tachycardia.


In [2]:
!pip install sacremoses sacrebleu

   ---------------------------------------- 0.0/897.5 kB ? eta -:--:--
   --------------------------------------- 897.5/897.5 kB 20.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    set_seed
)
from functools import partial # Not needed with the reassignment approach, but good practice if applicable

################################################################################
# 1. Load and Inspect Dataset
################################################################################

# Update these paths with your actual CSV locations
base_path = "./data/multiCochrane_all/filtered (r=0.5)/en"
data_files = {
    "train": os.path.join(base_path, "train0.5_en.csv"),
    "validation": os.path.join(base_path, "val0.5_en.csv"),
    "test": os.path.join(base_path, "test0.5_en.csv")
}

raw_datasets = load_dataset("csv", data_files=data_files)

################################################################################
# 2. Setup Model & Tokenizer
################################################################################

model_checkpoint = "google-t5/t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)

prefix = "Simplify: "
max_input_length = 256
max_target_length = 256

################################################################################
# 3. Preprocessing and Tokenization
################################################################################

def preprocess_function(examples):
    sources = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        [prefix + s for s in sources],
        max_length=max_input_length,
        padding="max_length", # Pad here initially
        truncation=True
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            padding="max_length", # Pad here initially
            truncation=True
        )["input_ids"]

    labels = [
        [(lbl if lbl != tokenizer.pad_token_id else -100) for lbl in seq]
        for seq in labels
    ]
    model_inputs["labels"] = labels
    # Keep the original source text needed for SARI computation later
    model_inputs["source_text"] = sources # Make sure this is added

    return model_inputs

# Apply tokenization
# Remove all original columns, preprocess_function returns what we need
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

# Ensure 'source_text' is present (it should be if preprocess_function added it)
# print(tokenized_datasets["train"].column_names) # Should include 'source_text'

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]
test_dataset = tokenized_datasets["test"]

################################################################################
# 4. Data Collator
################################################################################

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    # The model argument is not needed for DataCollatorForSeq2Seq with padding=True/max_length in tokenizer
    # model=model # -> Remove this if padding is handled by tokenizer/map
    # Let the collator handle padding dynamically based on the batch if you didn't pad in preprocess
    # If you padded to max_length in preprocess, this just converts lists to tensors.
)

################################################################################
# 5. Define the Metric & compute_metrics Function FOR VALIDATION
################################################################################

sari_metric = evaluate.load("sari")

# --- Function used during training for VALIDATION dataset ---
def compute_metrics_eval(eval_preds):
    """
    Computes SARI for the EVALUATION (validation) set.
    Assumes 'eval_dataset' global variable holds the validation data.
    """
    predictions, labels = eval_preds

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # *** Uses the global eval_dataset for sources ***
    sources = eval_dataset["source_text"]

    # Check lengths just before computation (optional debug)
    # print(f"Eval Sources: {len(sources)}, Preds: {len(decoded_preds)}, Labels: {len(decoded_labels)}")

    # Ensure lengths match - if not, something else is wrong upstream
    if len(sources) != len(decoded_preds):
         raise ValueError(f"EVAL: Mismatch in sources ({len(sources)}) and predictions ({len(decoded_preds)})")

    all_refs = [[lbl] for lbl in decoded_labels]

    sari_results = sari_metric.compute(
        sources=sources,
        predictions=decoded_preds,
        references=all_refs
    )
    return {"sari": sari_results["sari"]}

# --- Function used after training for TEST dataset ---
def compute_metrics_test(eval_preds):
    """
    Computes SARI for the TEST set.
    Assumes 'test_dataset' global variable holds the test data.
    """
    predictions, labels = eval_preds

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # *** Uses the global test_dataset for sources ***
    sources = test_dataset["source_text"] # <-- The key change!

    # Check lengths just before computation (optional debug)
    # print(f"Test Sources: {len(sources)}, Preds: {len(decoded_preds)}, Labels: {len(decoded_labels)}")

     # Ensure lengths match - if not, something else is wrong upstream
    if len(sources) != len(decoded_preds):
         raise ValueError(f"TEST: Mismatch in sources ({len(sources)}) and predictions ({len(decoded_preds)})")

    all_refs = [[lbl] for lbl in decoded_labels]

    sari_results = sari_metric.compute(
        sources=sources,
        predictions=decoded_preds,
        references=all_refs
    )
    # Important: Trainer expects metric keys to start with 'eval_' by default.
    # If you want distinct names, use metric_key_prefix in evaluate() call.
    # Here we return 'sari' and rely on the prefix if needed.
    return {"sari": sari_results["sari"]}


################################################################################
# 6. Set Training Arguments
################################################################################

set_seed(42)

training_args = Seq2SeqTrainingArguments(
    output_dir="./content_simplifier_t5_small_simple_trainer_SARI/",
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=6,
    learning_rate=5e-5,
    per_device_train_batch_size=32, # Adjust based on GPU memory
    per_device_eval_batch_size=32, # Adjust based on GPU memory
    gradient_accumulation_steps=4,
    predict_with_generate=True,
    generation_max_length=max_target_length,
    logging_dir='./logs',             # Directory for logs
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="sari",
    greater_is_better=True,
    fp16=torch.cuda.is_available(), # Enable only if supported and CUDA is available
    # Removed include_inputs_for_metrics as it's not the chosen solution here
)

################################################################################
# 7. Initialize the Trainer
################################################################################

model = T5ForConditionalGeneration.from_pretrained(model_checkpoint)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    # Use the evaluation function designed for the validation set during training
    compute_metrics=compute_metrics_eval
)

################################################################################
# 8. Train the Model
################################################################################

print("Starting training...")
train_result = trainer.train()
# Save the final best model
trainer.save_model(training_args.output_dir)
print("Training finished.")

################################################################################
# 9. Evaluate on Test Set
################################################################################

print("Evaluating on the test set...")
# *** Temporarily set the compute_metrics function to the one for the test set ***
trainer.compute_metrics = compute_metrics_test
# Run evaluation on the test dataset
# The trainer will automatically add "eval_" prefix to the metric keys.
test_metrics = trainer.evaluate(test_dataset)

# *** Optional: Restore the original compute_metrics if you plan to use the trainer further ***
# trainer.compute_metrics = compute_metrics_eval

# Print the test metric (key will be 'eval_sari')
print(f"Test set SARI: {test_metrics['eval_sari']:.4f}")
# You can also save test metrics if needed
# For example:
# with open(os.path.join(training_args.output_dir, "test_results.json"), "w") as f:
#    json.dump(test_metrics, f, indent=4)

################################################################################
# 10. Inference Example
################################################################################

# Ensure the model used for inference is the trained one (load_best_model_at_end should handle this)
# If not sure, explicitly load:
# model = T5ForConditionalGeneration.from_pretrained(training_args.output_dir).to("cuda" if torch.cuda.is_available() else "cpu")
# tokenizer = T5Tokenizer.from_pretrained(training_args.output_dir)



c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\training_args.py:1612: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Ruben\AppData\Local\Temp\ipykernel_111200\2400955123.py:211: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting training...


Epoch,Training Loss,Validation Loss,Sari
1,1.726600,2.560388,43.132052
2,1.673800,2.502180,43.323753
3,1.637200,2.474097,42.790828
4,1.607700,2.453658,43.268430
5,1.586900,2.442612,43.898942


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training finished.
Evaluating on the test set...


Test set SARI: 41.3155


In [22]:
print("\nRunning inference example...")
sample_text = (
    "The prevalence of adverse renal effects ranged from 0% to 84%."
)

prompt = prefix + sample_text
# Use the trainer's device placement logic
device = trainer.model.device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_ids = trainer.model.generate(
        **inputs,
        max_length=256,
        do_sample=True,       # Enable sampling to avoid replication
        top_p=0.95,           # Use top-p sampling for diversity
        num_beams=1           # No beam search
    )
summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Original:", sample_text)
print("Simplified:", summary)


Running inference example...
Original: The prevalence of adverse renal effects ranged from 0% to 84%.
Simplified: The prevalence of adverse renal effects ranged from 0% to 84%.
